In [1]:
# Some sample code for importing model which has been downloaded from the HEC

from problems.pdtrp.problem_pdtrp import PDTRPDataset
from problems.pdcvrp.problem_pdcvrp import PDCVRPDataset
from problems.pdtrp_tw.problem_pdtrptw import PDTRPTWDataset
from problems.pdcvrp_tw.problem_pdcvrptw import PDCVRPTWDataset

from torch.utils.data import DataLoader

from utils import load_model, move_to
from train import set_decode_type
import torch
from tqdm import tqdm

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")

model_filepath = 'models_HEC/rl-ar-var-20pnn-gnn-max-pdtrp_20250206T205558'

model, model_args = load_model(model_filepath)

model.to(device)
set_decode_type(model, "greedy")
model.eval()

/home/trainer1/learning-pdtsp/utils/functions.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(load_path, map_location=lambda storage, loc: storage)  #


Loading model from models_HEC/rl-ar-var-20pnn-gnn-max-pdtrp_20250206T205558/epoch-99.pt


/home/trainer1/learning-pdtsp/utils/functions.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  load_data = torch.load(


DynamicAttentionModel(
  (init_embed_depot): Linear(in_features=2, out_features=128, bias=True)
  (node_embed): Linear(in_features=2, out_features=128, bias=True)
  (edge_embed): Embedding(2, 128)
  (embedder): GNNEncoder(
    (layers): ModuleList(
      (0-2): 3 x GNNLayer(
        (U): Linear(in_features=128, out_features=128, bias=True)
        (V): Linear(in_features=128, out_features=128, bias=True)
        (A): Linear(in_features=128, out_features=128, bias=True)
        (B): Linear(in_features=128, out_features=128, bias=True)
        (C): Linear(in_features=128, out_features=128, bias=True)
        (norm_h): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
        (norm_e): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      )
    )
  )
  (project_node_embeddings): Linear(in_features=128, out_features=384, bias=False)
  (project_fixed_context): Linear(in_features=128, out_features=128, bias=False)
  (project_s

In [2]:
# Function for plotting solutions
%matplotlib inline
from matplotlib import pyplot as plt

from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import numpy as np

# Code inspired by Google OR Tools plot:
# https://github.com/google/or-tools/blob/fb12c5ded7423d524fc6c95656a9bdc290a81d4d/examples/python/cvrptw_plot.py

def plot_tsp(xy, tour, ax1):
    """
    Plot the TSP tour on matplotlib axis ax1.
    """
    
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    xs, ys = xy[tour].transpose()
    xs, ys = xy[tour].transpose()
    dx = np.roll(xs, -1) - xs
    dy = np.roll(ys, -1) - ys
    d = np.sqrt(dx * dx + dy * dy)
    lengths = d.cumsum()
    
    # Scatter nodes
    ax1.scatter(xs, ys, s=40, color='blue')
    # Starting node
    ax1.scatter([xs[0]], [ys[0]], s=100, color='red')

    #label nodes
    for i, (x, y) in enumerate(xy):
        ax1.text(x, y, str(i), fontsize=12, color='green')
    
    # Arcs
    qv = ax1.quiver(
        xs, ys, dx, dy,
        scale_units='xy',
        angles='xy',
        scale=1,
    )
    
    ax1.set_title('{} nodes, total length {:.2f}'.format(len(tour), lengths[-1]))



In [14]:
problem_type = 'PDTRP'
NUM_SAMPLES = 128
BATCH_SIZE = 16

filename = 'data/pdtrp/test_data_PDTRP.txt'

# SET SEEDS FOR REPRODUCIBILITY
np.random.seed(2512)
torch.manual_seed(2512)

if problem_type == 'PDTRP':
    dataset = PDTRPDataset(num_samples=NUM_SAMPLES, batch_size=BATCH_SIZE, filename=filename)
elif problem_type == 'PDCVRP':
    dataset = PDCVRPDataset(num_samples=NUM_SAMPLES, batch_size=BATCH_SIZE, filename=filename)
elif problem_type == 'PDTRPTW':
    dataset = PDTRPTWDataset(num_samples=NUM_SAMPLES, batch_size=BATCH_SIZE, filename=filename)
elif problem_type == 'PDCVRPTW':
    dataset = PDCVRPTWDataset(num_samples=NUM_SAMPLES, batch_size=BATCH_SIZE, filename=filename)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

Loading from data/pdtrp/test_data_PDTRP.txt...


100%|##########| 128/128 [00:00<00:00, 4985.43it/s]


In [17]:
with torch.no_grad():

    for bat in tqdm(dataloader):
        
        if problem_type == 'PDTRP':
            input = {
                'loc': move_to(bat['all_nodes'], device),
                'service_times': move_to(bat['service_times'], device),
                'arrival_times': move_to(bat['arrival_times'], device),
                'graph': move_to(bat['graph'], device)
            }

        cost, _, pi = model(input, return_pi=True)

        pi_with_depot = torch.cat((torch.zeros(pi.size(0), 1, dtype=torch.long), pi), dim=1)

        or_tools_tour = bat['output'][0].cpu().numpy()
        break

  0%|          | 0/8 [00:00<?, ?it/s]

{'all_nodes': tensor([[[0.5000, 0.5000],
         [0.2480, 0.1960],
         [0.3840, 0.1240],
         ...,
         [0.5800, 0.3590],
         [0.6430, 0.4690],
         [0.4040, 0.1440]],

        [[0.5000, 0.5000],
         [0.4950, 0.1340],
         [0.7450, 0.0880],
         ...,
         [0.4960, 0.1720],
         [0.6100, 0.1440],
         [0.4790, 0.5820]],

        [[0.5000, 0.5000],
         [0.0260, 0.2900],
         [0.6330, 0.1180],
         ...,
         [0.0170, 0.5230],
         [0.9230, 0.5990],
         [0.9880, 0.3730]],

        ...,

        [[0.5000, 0.5000],
         [0.1740, 0.2030],
         [0.3060, 0.0830],
         ...,
         [0.1970, 0.4260],
         [0.0200, 0.6010],
         [0.5720, 0.9560]],

        [[0.5000, 0.5000],
         [0.0120, 0.0230],
         [0.7100, 0.1400],
         ...,
         [0.6730, 0.3670],
         [0.8930, 0.4310],
         [0.4780, 0.6960]],

        [[0.5000, 0.5000],
         [0.2920, 0.1640],
         [0.3320, 0.2590],
 

  0%|          | 0/8 [02:05<?, ?it/s]


KeyboardInterrupt: 